# 02 - Exploratory Data Analysis & Advanced Prototypes

This notebook performs exploration on the **Retail Multi-Brand** dataset.  
Beyond basic profiling, we also prototype **Advanced Analytics** that inform our dbt model design.

**Goals:**
- Validate data distributions and quality (Profiling).
- Prototype **Market Basket Analysis** (Product Affinity).
- Prototype **Customer Retention** (Cohort Analysis).
- Prototype **Inventory Turnover** insights.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os

sns.set(style="whitegrid")

# Setup Paths
BASE_DIR = os.path.dirname(os.getcwd())
RAW_DIR = os.path.join(BASE_DIR, 'data', 'raw')

def safe_read(fname):
    path = os.path.join(RAW_DIR, fname)
    if os.path.exists(path):
        return pd.read_csv(path)
    print(f"Warning: {fname} not found!")
    return None

## 1. Data Profiling & Cleaning
We load our core datasets and check for missing values.

In [ ]:
df_customers = safe_read("raw_customers.csv")
df_products = safe_read("raw_products.csv")
df_sales = safe_read("raw_sales.csv")

if df_sales is not None:
    print(f"Sales dataset: {df_sales.shape[0]} rows")
    print(df_sales.head(3))

## 2. Advanced Prototype: Market Basket Analysis
Which products do customers frequently buy together? This logic will later be standardized in dbt/BI.

In [ ]:
from collections import Counter
from itertools import combinations

def prototype_mba(sales_df):
    # Group product IDs by transaction (using date+customer as proxy for basket)
    baskets = sales_df.groupby(['customer_id', 'date'])['product_id'].apply(list)
    
    pairs = Counter()
    for items in baskets:
        if len(items) > 1:
            pairs.update(combinations(sorted(set(items)), 2))
            
    return pd.DataFrame(pairs.most_common(10), columns=['Product Pair', 'Frequency'])

if df_sales is not None:
    top_pairs = prototype_mba(df_sales)
    print("Top 10 Product Pairs:")
    print(top_pairs)

## 3. Advanced Prototype: Cohort Analysis (Retention)
Tracking how many customers return month-over-month.

In [ ]:
if df_sales is not None:
    df_sales['date'] = pd.to_datetime(df_sales['date'])
    df_sales['order_month'] = df_sales['date'].dt.to_period('M')
    
    # First purchase month per customer
    df_sales['cohort'] = df_sales.groupby('customer_id')['date'].transform('min').dt.to_period('M')
    
    cohort_data = df_sales.groupby(['cohort', 'order_month']).agg(n_customers=('customer_id', 'nunique')).reset_index()
    cohort_pivot = cohort_data.pivot(index='cohort', columns='order_month', values='n_customers')
    
    plt.figure(figsize=(12, 6))
    sns.heatmap(cohort_pivot, annot=True, fmt='.0f', cmap='YlGnBu')
    plt.title("Customer Retention Cohorts (Monthly)")
    plt.show()

## Summary of Findings
- **Data Quality**: Ingestion paths are now robustly handled.
- **Insight Potential**: Prototyping MBA and Cohorts shows high potential for strategic business dashboards.
- **Next Steps**: These logics are being moved into `dbt` models (stg -> int -> marts) for production scalability.